# Exercise 3 Solution: JOINs and Relationships

Complete solutions for JOIN operations and multi-table queries.

In [ ]:
import duckdb
conn = duckdb.connect()

## Setup: Load Data

First, load the required AdventureWorks tables:

In [ ]:
# Product tables
conn.execute("""
    CREATE TABLE products AS 
    SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.Product.csv')
""")

conn.execute("""
    CREATE TABLE product_categories AS 
    SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.ProductCategory.csv')
""")

conn.execute("""
    CREATE TABLE product_subcategories AS 
    SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.ProductSubcategory.csv')
""")

print("✓ Tables loaded")

## Task 1: Understand Data Structure

a) Show the structure of the `products` table

In [ ]:
# Solution
conn.execute("DESCRIBE products").df()

b) Show the structure of the `product_categories` table

In [ ]:
# Solution
conn.execute("DESCRIBE product_categories").df()

c) Show the structure of the `product_subcategories` table

In [ ]:
# Solution
conn.execute("DESCRIBE product_subcategories").df()

## Task 2: Simple JOIN

Join `products` with `product_subcategories` and show:
- Product name
- Subcategory name
- List price

Show only the first 10 results.

In [ ]:
# Solution
conn.execute("""
    SELECT 
        p.Name AS product_name,
        ps.Name AS subcategory_name,
        p.ListPrice
    FROM products p
    JOIN product_subcategories ps 
        ON p.ProductSubcategoryID = ps.ProductSubcategoryID
    LIMIT 10
""").df()

## Task 3: Multiple JOINs

Create a query that connects all three tables and shows:
- Product name
- Subcategory name
- Category name
- List price

Sort by category and price descending.

In [ ]:
# Solution
conn.execute("""
    SELECT 
        p.Name AS product_name,
        ps.Name AS subcategory_name,
        pc.Name AS category_name,
        p.ListPrice
    FROM products p
    JOIN product_subcategories ps 
        ON p.ProductSubcategoryID = ps.ProductSubcategoryID
    JOIN product_categories pc 
        ON ps.ProductCategoryID = pc.ProductCategoryID
    ORDER BY pc.Name, p.ListPrice DESC
""").df()

## Task 4: Aggregation with JOINs

a) How many products are there per category?

In [ ]:
# Solution
conn.execute("""
    SELECT 
        pc.Name AS category_name,
        COUNT(*) AS product_count
    FROM products p
    JOIN product_subcategories ps 
        ON p.ProductSubcategoryID = ps.ProductSubcategoryID
    JOIN product_categories pc 
        ON ps.ProductCategoryID = pc.ProductCategoryID
    GROUP BY pc.Name
    ORDER BY product_count DESC
""").df()

b) What is the average list price per category?

In [ ]:
# Solution
conn.execute("""
    SELECT 
        pc.Name AS category_name,
        ROUND(AVG(p.ListPrice), 2) AS avg_price
    FROM products p
    JOIN product_subcategories ps 
        ON p.ProductSubcategoryID = ps.ProductSubcategoryID
    JOIN product_categories pc 
        ON ps.ProductCategoryID = pc.ProductCategoryID
    GROUP BY pc.Name
    ORDER BY avg_price DESC
""").df()

c) Find the most expensive product in each category

In [ ]:
# Solution
conn.execute("""
    WITH ranked_products AS (
        SELECT 
            pc.Name AS category_name,
            p.Name AS product_name,
            p.ListPrice,
            ROW_NUMBER() OVER (PARTITION BY pc.Name ORDER BY p.ListPrice DESC) AS rn
        FROM products p
        JOIN product_subcategories ps 
            ON p.ProductSubcategoryID = ps.ProductSubcategoryID
        JOIN product_categories pc 
            ON ps.ProductCategoryID = pc.ProductCategoryID
    )
    SELECT category_name, product_name, ListPrice
    FROM ranked_products
    WHERE rn = 1
    ORDER BY ListPrice DESC
""").df()

## Task 5: Sales Analysis

Load additional tables and analyze sales data:

In [ ]:
# Load sales tables
conn.execute("""
    CREATE TABLE sales_orders AS 
    SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesOrderHeader.csv')
""")

conn.execute("""
    CREATE TABLE sales_details AS 
    SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesOrderDetail.csv')
""")

conn.execute("""
    CREATE TABLE territories AS 
    SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesTerritory.csv')
""")

print("✓ Sales tables loaded")

a) Show the top 10 products by sales quantity (OrderQty)

In [ ]:
# Solution
conn.execute("""
    SELECT 
        p.Name AS product_name,
        SUM(sd.OrderQty) AS total_quantity,
        SUM(sd.LineTotal) AS total_revenue
    FROM sales_details sd
    JOIN products p ON sd.ProductID = p.ProductID
    GROUP BY p.Name
    ORDER BY total_quantity DESC
    LIMIT 10
""").df()

b) Which category has the highest total revenue (LineTotal)?

In [ ]:
# Solution
conn.execute("""
    SELECT 
        pc.Name AS category_name,
        SUM(sd.LineTotal) AS total_revenue,
        COUNT(DISTINCT sd.SalesOrderID) AS order_count
    FROM sales_details sd
    JOIN products p ON sd.ProductID = p.ProductID
    JOIN product_subcategories ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
    JOIN product_categories pc ON ps.ProductCategoryID = pc.ProductCategoryID
    GROUP BY pc.Name
    ORDER BY total_revenue DESC
""").df()

## Bonus Task

Create a comprehensive analysis showing:
- Territory name
- Number of orders
- Total revenue
- Average order value
- Top-selling product category in that territory

This requires multiple JOINs and window functions!

In [ ]:
# Solution
conn.execute("""
    WITH territory_stats AS (
        SELECT 
            t.Name AS territory_name,
            COUNT(DISTINCT so.SalesOrderID) AS order_count,
            SUM(so.TotalDue) AS total_revenue,
            ROUND(AVG(so.TotalDue), 2) AS avg_order_value
        FROM sales_orders so
        JOIN territories t ON so.TerritoryID = t.TerritoryID
        GROUP BY t.Name
    ),
    territory_categories AS (
        SELECT 
            t.Name AS territory_name,
            pc.Name AS category_name,
            SUM(sd.LineTotal) AS category_revenue,
            ROW_NUMBER() OVER (PARTITION BY t.Name ORDER BY SUM(sd.LineTotal) DESC) AS rn
        FROM sales_orders so
        JOIN territories t ON so.TerritoryID = t.TerritoryID
        JOIN sales_details sd ON so.SalesOrderID = sd.SalesOrderID
        JOIN products p ON sd.ProductID = p.ProductID
        JOIN product_subcategories ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
        JOIN product_categories pc ON ps.ProductCategoryID = pc.ProductCategoryID
        GROUP BY t.Name, pc.Name
    )
    SELECT 
        ts.territory_name,
        ts.order_count,
        ts.total_revenue,
        ts.avg_order_value,
        tc.category_name AS top_category
    FROM territory_stats ts
    JOIN territory_categories tc ON ts.territory_name = tc.territory_name AND tc.rn = 1
    ORDER BY ts.total_revenue DESC
""").df()

## 🎉 Congratulations!

You now master JOINs in DuckDB!